# 00 — Data Pipeline: Ingestion and Memory Benchmark

Turns the raw Milan CDR archive (`data/raw/sms-call-internet-mi-YYYY-MM-DD.txt`,
~10GB across 30 daily files) into the small, analysis-ready artifacts every
later notebook depends on, and demonstrates *why* the loading strategy used
here is necessary on an 8GB RAM machine.

Each daily raw file is tab-separated, headerless, one row per
`(square_id, time_interval, country_code)`:

```
square_id  timestamp_ms  country_code  sms_in  sms_out  call_in  call_out  internet
```

A given `(square, 10-minute interval)` is split across many rows (one per
active country code), so raw row counts are ~20-30x the `(square, interval)`
pairs actually needed. This notebook focuses on the `internet` column only.

**Produces:**
- `data/processed/square_totals.csv` — total internet traffic per square (all 10,000 squares)
- `data/processed/target_squares.yaml` — which squares were selected and why
- `data/processed/target_squares_timeseries.csv` — full 10-min series for the target squares
- `results/memory_benchmark.csv` — naive vs. optimized peak-memory comparison

In [1]:
import glob
import os
import subprocess
import sys
import time

import pandas as pd
import yaml

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import common
from common import (
    CHUNKSIZE, FIXED_SQUARES, OBSERVATION_START, OBSERVATION_END,
    RAW_DIR, RAW_GLOB, SQUARE_TOTALS_PATH, TARGET_SERIES_PATH, TARGET_SQUARES_META_PATH,
)

common.set_seed()
os.makedirs("data/processed", exist_ok=True)
os.makedirs("results", exist_ok=True)

## 1. Memory-efficient ingestion strategy

Daily files are 300-400MB each; loading the whole archive naively (all 8
columns, default `int64`/`float64` dtypes, no aggregation) is not viable on
8GB RAM. The strategy is a **two-pass, chunked, column-pruned, dtype-downcast
aggregation**:

- **Pass 1** (`compute_square_totals`): stream every file in fixed-size
  chunks, reading only the 2 columns needed, collapsing country codes with a
  per-chunk `groupby(square_id).sum()`. Only a 10,000-length running total is
  kept in memory for the whole archive.
- **Pass 2** (`extract_target_series`): once the top-traffic squares are known
  from Pass 1, stream the files again reading 3 columns, filtering to the
  ~5 target squares *before* aggregating (>99.9% row reduction), then
  collapsing country codes per `(square, timestamp)`.

This keeps peak memory bounded by one chunk, never by file or archive size.

In [2]:
RAW_COLUMN_NAMES = [
    "square_id", "timestamp_ms", "country_code",
    "sms_in", "sms_out", "call_in", "call_out", "internet",
]
COL_INDEX = {name: i for i, name in enumerate(RAW_COLUMN_NAMES)}


def list_raw_files(raw_dir: str, raw_glob: str):
    files = sorted(glob.glob(os.path.join(raw_dir, raw_glob)))
    if not files:
        raise FileNotFoundError(f"No raw files matching '{raw_glob}' found in '{raw_dir}'.")
    return files


def filter_files_by_date(files, start: str, end: str):
    """Keep only files whose embedded YYYY-MM-DD date falls within [start, end]."""
    start_ts, end_ts = pd.Timestamp(start), pd.Timestamp(end)
    kept = []
    for f in files:
        date_str = os.path.basename(f).replace("sms-call-internet-mi-", "").replace(".txt", "")
        try:
            file_date = pd.Timestamp(date_str)
        except ValueError:
            continue
        if start_ts <= file_date <= end_ts:
            kept.append(f)
    return kept


def compute_square_totals(files, chunksize: int = 2_000_000) -> pd.Series:
    """Pass 1: total internet traffic per square_id across the given files."""
    totals = pd.Series(dtype="float64")
    usecols = [COL_INDEX["square_id"], COL_INDEX["internet"]]
    dtypes = {COL_INDEX["square_id"]: "int32", COL_INDEX["internet"]: "float32"}

    for path in files:
        for chunk in pd.read_csv(
            path, sep="\t", header=None, usecols=usecols, dtype=dtypes,
            names=["square_id", "internet"], chunksize=chunksize,
        ):
            chunk_sum = chunk.groupby("square_id", sort=False)["internet"].sum()
            totals = totals.add(chunk_sum, fill_value=0.0)

    totals.index.name = "square_id"
    return totals.sort_index()


def extract_target_series(files, target_squares, chunksize: int = 2_000_000) -> pd.DataFrame:
    """Pass 2: full 10-minute internet time series for a handful of squares."""
    target_squares = set(int(s) for s in target_squares)
    usecols = [COL_INDEX["square_id"], COL_INDEX["timestamp_ms"], COL_INDEX["internet"]]
    dtypes = {
        COL_INDEX["square_id"]: "int32",
        COL_INDEX["timestamp_ms"]: "int64",
        COL_INDEX["internet"]: "float32",
    }

    day_frames = []
    for path in files:
        chunk_frames = []
        for chunk in pd.read_csv(
            path, sep="\t", header=None, usecols=usecols, dtype=dtypes,
            names=["square_id", "timestamp_ms", "internet"], chunksize=chunksize,
        ):
            chunk = chunk[chunk["square_id"].isin(target_squares)]
            if len(chunk):
                chunk_frames.append(
                    chunk.groupby(["square_id", "timestamp_ms"], sort=False)["internet"].sum().reset_index()
                )
        if chunk_frames:
            day_df = pd.concat(chunk_frames, ignore_index=True)
            day_df = day_df.groupby(["square_id", "timestamp_ms"], sort=False)["internet"].sum().reset_index()
            day_frames.append(day_df)

    result = pd.concat(day_frames, ignore_index=True)
    result["timestamp"] = pd.to_datetime(result["timestamp_ms"], unit="ms")
    result = result.drop(columns="timestamp_ms").sort_values(["square_id", "timestamp"])
    return result.reset_index(drop=True)

## 2. Pass 1 — total traffic per square, and target-square selection

In [3]:
all_files = list_raw_files(RAW_DIR, RAW_GLOB)
obs_files = filter_files_by_date(all_files, OBSERVATION_START, OBSERVATION_END)
print(f"Observation period {OBSERVATION_START} -> {OBSERVATION_END}: "
      f"{len(obs_files)} files selected out of {len(all_files)} in {RAW_DIR}/")

t0 = time.perf_counter()
totals = compute_square_totals(obs_files, chunksize=CHUNKSIZE)
pass1_elapsed = time.perf_counter() - t0
print(f"Pass 1 done in {pass1_elapsed:.1f}s over {len(obs_files)} files -> {len(totals)} squares")
totals.to_csv(SQUARE_TOTALS_PATH, header=["internet_total"])

Observation period 2013-11-01 -> 2013-11-30: 30 files selected out of 62 in data/raw/
Pass 1 done in 39.2s over 30 files -> 10000 squares


In [4]:
top3 = totals.sort_values(ascending=False).head(3)
target_squares = sorted(set(int(s) for s in top3.index) | set(int(s) for s in FIXED_SQUARES))
print("Top-3 squares by total internet traffic:", top3.to_dict())
print("Target squares for time-series extraction (top-3 U fixed):", target_squares)

with open(TARGET_SQUARES_META_PATH, "w") as f:
    yaml.safe_dump({
        "top3_squares": [int(s) for s in top3.index],
        "top3_totals": {int(k): float(v) for k, v in top3.items()},
        "fixed_squares": [int(s) for s in FIXED_SQUARES],
        "target_squares": target_squares,
        "observation_period": {"start": OBSERVATION_START, "end": OBSERVATION_END},
    }, f)

Top-3 squares by total internet traffic: {5161: 6329187.671875, 5059: 5834589.46875, 5259: 5759423.14453125}
Target squares for time-series extraction (top-3 U fixed): [4159, 4556, 5059, 5161, 5259]


## 3. Pass 2 — full 10-minute series for the target squares only

In [ ]:
t0 = time.perf_counter()
series_df = extract_target_series(obs_files, target_squares, chunksize=CHUNKSIZE)
pass2_elapsed = time.perf_counter() - t0
print(f"Pass 2 done in {pass2_elapsed:.1f}s -> {len(series_df)} rows for {series_df['square_id'].nunique()} squares")
series_df.to_csv(TARGET_SERIES_PATH, index=False)
print(f"Saved: {SQUARE_TOTALS_PATH}, {TARGET_SERIES_PATH}, {TARGET_SQUARES_META_PATH}")
print(f"Combined Pass 1 + Pass 2 time: {pass1_elapsed + pass2_elapsed:.1f}s")

## 4. Memory benchmark: naive vs. optimized single-file load

To make the "memory-efficient" claim above concrete, this measures **peak
resident memory** for two ways of loading a single day file:
- **naive**: `pandas.read_csv`, all 8 columns, default `int64`/`float64` dtypes, no chunking, no aggregation
- **optimized**: chunked (2M rows/chunk), 2 columns only, `int32`/`float32`, aggregated per chunk

Each variant is run in its own subprocess (via a small throwaway script,
measured out-of-process with `/usr/bin/time -l`) so peak RSS reflects only
that code path, not this notebook's kernel or the other variant.

In [ ]:
BENCH_FILE = f"{RAW_DIR}/sms-call-internet-mi-2013-11-01.txt"

NAIVE_SNIPPET = f'''
import pandas as pd
RAW_COLUMN_NAMES = {RAW_COLUMN_NAMES!r}
df = pd.read_csv("{{path}}", sep="\\t", header=None, names=RAW_COLUMN_NAMES)
print(f"rows={{len(df)}} cols={{df.shape[1]}}")
'''.format(path=BENCH_FILE)

OPTIMIZED_SNIPPET = f'''
import pandas as pd
usecols, dtypes = [0, 7], {{0: "int32", 7: "float32"}}
chunk_sums = []
for chunk in pd.read_csv("{{path}}", sep="\\t", header=None, usecols=usecols, dtype=dtypes,
                          names=["square_id", "internet"], chunksize={CHUNKSIZE}):
    chunk_sums.append(chunk.groupby("square_id", sort=False)["internet"].sum())
out = pd.concat(chunk_sums, axis=1).sum(axis=1).reset_index()
print(f"rows={{len(out)}} cols={{out.shape[1]}}")
'''.format(path=BENCH_FILE)


def run_benchmark(label: str, snippet: str, script_path: str) -> dict:
    with open(script_path, "w") as f:
        f.write(snippet)
    t0 = time.perf_counter()
    result = subprocess.run(
        ["/usr/bin/time", "-l", sys.executable, script_path],
        capture_output=True, text=True,
    )
    elapsed = time.perf_counter() - t0
    os.remove(script_path)

    peak_rss_bytes = None
    for line in result.stderr.splitlines():
        if "maximum resident set size" in line:
            peak_rss_bytes = int(line.strip().split()[0])
    if peak_rss_bytes is None:
        raise RuntimeError(f"Could not parse peak RSS from /usr/bin/time output:\n{result.stderr}")

    rows_out, cols_out = None, None
    for line in result.stdout.splitlines():
        if line.startswith("rows="):
            parts = dict(p.split("=") for p in line.split())
            rows_out, cols_out = int(parts["rows"]), int(parts["cols"])

    return {
        "approach": label, "file": os.path.basename(BENCH_FILE),
        "rows_out": rows_out, "cols_out": cols_out, "elapsed_s": elapsed,
        "peak_rss_mb": peak_rss_bytes / (1024 ** 2),
    }


naive_result = run_benchmark("naive_full_load", NAIVE_SNIPPET, "_bench_naive.py")
optimized_result = run_benchmark("optimized_chunked_typed_aggregated", OPTIMIZED_SNIPPET, "_bench_optimized.py")

reduction_pct = 100 * (1 - optimized_result["peak_rss_mb"] / naive_result["peak_rss_mb"])
naive_result["peak_rss_reduction_pct"] = 0.0
optimized_result["peak_rss_reduction_pct"] = reduction_pct

bench_df = pd.DataFrame([naive_result, optimized_result])
bench_df.to_csv("results/memory_benchmark.csv", index=False)
bench_df

**Reading the result:** the optimized loader should show a large reduction
in peak RSS relative to the naive loader (column pruning + `int32`/`float32`
downcasting + per-chunk aggregation vs. loading all 8 columns at default
`int64`/`float64` width with no aggregation), while producing 10,000
aggregated rows instead of ~4.8 million raw rows. Applied end-to-end across
the full archive (Pass 1 + Pass 2 above), both passes together process ~10GB
in well under two minutes, with memory bounded by chunk size rather than
file or archive size throughout.